# Nepali-English Machine Translation using Qwen 2.5

This notebook fine-tunes Qwen2.5-0.5B-Instruct using LoRA and TRL's SFTTrainer on a Nepali-English parallel dataset. Checkpoints are stored in Google Drive to allow training to resume after Colab disconnects.

## 1. Environment Setup

In [ ]:
# Cell 1 — installs
!pip install -q -U transformers accelerate trl peft datasets huggingface_hub sacrebleu torchao

## 2. Authentication & Storage

In [ ]:
# Cell 2 — Hugging Face login (needs write access to sjcMLT for later pushing)
from huggingface_hub import login
login()


In [ ]:
# Cell 3 — mount Drive (checkpoints persist here across disconnects)
from google.colab import drive
drive.mount("/content/drive")


## 3. Loading Dataset

In [ ]:
# Cell 4 — load the dataset from the Hub (already has leakage-safe train/validation split)
from datasets import load_dataset

DATASET_REPO = "sjcMLT/nep-eng-parallel-corpus"

dataset_dict = load_dataset(DATASET_REPO)
train_raw = dataset_dict["train"]
val_raw = dataset_dict["validation"]

print(train_raw)
print(val_raw)


## 4. Tokenization & Chat Formatting

In [ ]:
# Cell 5 — load the tokenizer
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"

print("pad_token:", tokenizer.pad_token, "| eos_token:", tokenizer.eos_token)
print("chat_template set:", tokenizer.chat_template is not None)


In [ ]:
# Cell 6 — build messages column for both splits
def build_messages(example):
    if example["direction"] == "nep_to_eng":
        instruction = f"Translate the following Nepali text to English: {example['source']}"
    else:
        instruction = f"Translate the following English text to Nepali: {example['source']}"

    return {
        "messages": [
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": example["target"]},
        ]
    }

train_dataset = train_raw.map(
    build_messages,
    num_proc=2,
    remove_columns=[c for c in train_raw.column_names if c != "direction"],
)
eval_dataset = val_raw.map(
    build_messages,
    num_proc=2,
    remove_columns=[c for c in val_raw.column_names if c != "direction"],
)

print(train_dataset)
print(eval_dataset)


In [ ]:
# Cell 7 — helper to load a fresh base model instance (used for both timing test and real run,so the timing test doesn't mutate the model we actually train on)
import torch
from transformers import AutoModelForCausalLM

BF16_SUPPORTED = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16_SUPPORTED else torch.float16
print("Using dtype:", DTYPE)

def load_base_model():
    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE)
    m.to("cuda")
    return m


## 5. Model & LoRA Configuration

In [ ]:
# Cell 8 — LoRA config (attention + MLP)
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)


## 5.1 Training Speed Benchmark (Optional)

In [ ]:
# after restart, before the timing test, also worth checking actual free memory to confirm a clean slate
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


In [ ]:
# Cell 9 (rerun) — self-contained, with imports included
import time
from trl import SFTConfig, SFTTrainer

timing_model = load_base_model()

timing_config = SFTConfig(
    output_dir="/content/timing_test",
    max_length=512,
    truncation_mode="keep_start",
    packing=False,
    assistant_only_loss=True,
    bf16=BF16_SUPPORTED,
    fp16=not BF16_SUPPORTED,
    gradient_checkpointing=False,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    optim="adamw_torch_fused",
    max_steps=50,
    logging_steps=10,
    eval_strategy="no",
    save_strategy="no",
    dataset_num_proc=2,
    seed=42,
    report_to="none",
)

timing_trainer = SFTTrainer(
    model=timing_model,
    args=timing_config,
    train_dataset=train_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

start = time.time()
timing_trainer.train()
elapsed = time.time() - start

steps_per_sec = 50 / elapsed
examples_per_sec = steps_per_sec * 16 * 8
print(f"Elapsed: {elapsed:.1f}s for 50 steps -> {steps_per_sec:.4f} steps/sec ({examples_per_sec:.1f} examples/sec)")

import gc
del timing_model, timing_trainer
gc.collect()
torch.cuda.empty_cache()


## 5.2 Training Time Estimation

In [ ]:
# Cell 10 — step budget from the confirmed timing result
TARGET_TRAINING_HOURS = 18
steps_per_sec = 0.16

MAX_STEPS = int(steps_per_sec * 3600 * TARGET_TRAINING_HOURS)
print("Planned max_steps:", MAX_STEPS)
print("Epoch coverage:", round(MAX_STEPS * 16 * 8 / len(train_dataset) * 100, 2), "%")


## 6. Training Configuration

In [ ]:
# Cell 11 — real SFTConfig (gradient_checkpointing=False, batch config matching the timing test)
EVAL_STEPS = 150

sft_config = SFTConfig(
    output_dir="/content/drive/MyDrive/qwen_translation_sft",
    max_length=512,
    truncation_mode="keep_start",
    packing=False,
    assistant_only_loss=True,
    bf16=BF16_SUPPORTED,
    fp16=not BF16_SUPPORTED,
    gradient_checkpointing=False,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    optim="adamw_torch_fused",
    max_steps=MAX_STEPS,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    per_device_eval_batch_size=16,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=5,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    dataset_num_proc=2,
    seed=42,
    report_to="tensorboard",
)

print(sft_config)


In [ ]:
# Cell 12 — load the real (fresh) model for training
model = load_base_model()


In [ ]:
# Cell 13 — build the real trainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.model.print_trainable_parameters()


In [ ]:
# Cell 14 — cross-entropy loss only (via SFTTrainer's built-in eval),
from transformers import TrainerCallback, EarlyStoppingCallback

class TranslationSampleCallback(TrainerCallback):
    def __init__(self, tokenizer, eval_dataset_raw, run_every_n_evals=3, n_samples_print=6):
        self.tokenizer = tokenizer
        self.eval_dataset_raw = eval_dataset_raw
        self.run_every_n_evals = run_every_n_evals
        self.n_samples_print = n_samples_print
        self._eval_count = 0

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        self._eval_count += 1
        if self._eval_count % self.run_every_n_evals != 0:
            return

        model.eval()
        subset = self.eval_dataset_raw.shuffle(seed=self._eval_count).select(range(self.n_samples_print))

        print(f"\n>>> Step {state.global_step} sample translations:")

        for row in subset:
            if row["direction"] == "nep_to_eng":
                instruction = f"Translate the following Nepali text to English: {row['source']}"
            else:
                instruction = f"Translate the following English text to Nepali: {row['source']}"

            messages = [{"role": "user", "content": instruction}]
            prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = self.tokenizer(prompt, return_tensors="pt").to(model.device)

            with torch.no_grad():
                output_ids = model.generate(**inputs, max_new_tokens=128, do_sample=False)

            generated = self.tokenizer.decode(
                output_ids[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True,
            )

            print(f"\n[{row['direction']}] SRC: {row['source']}")
            print(f"REF: {row['target']}")
            print(f"GEN: {generated}")

        model.train()


early_stopping = EarlyStoppingCallback(early_stopping_patience=5)
sample_check = TranslationSampleCallback(tokenizer=tokenizer, eval_dataset_raw=val_raw)

trainer.add_callback(early_stopping)
trainer.add_callback(sample_check)


## 7. Fine-Tuning

In [ ]:
#cell 15 - start training
trainer.train()

## 8. Resume Training (Optional)

In [ ]:
# Cell 16 — if resuming after a disconnect, use this instead of Cell 15
trainer.train(resume_from_checkpoint=True)


In [ ]:
trainer.save_model("/content/drive/MyDrive/qwen-lora-final")
tokenizer.save_pretrained("/content/drive/MyDrive/qwen-lora-final")